# MIC Prediction — Modeling Notebook

Predicting minimum inhibitory concentration (MIC) of antimicrobial peptides from
sequence, physicochemical descriptors, and AlphaFold2-derived structural features.

**What this notebook is.** A cleaned, linear presentation of the final modeling path:
sequence encoder → hybrid (sequence + physicochemical) model → ensemble →
out-of-distribution evaluation on 69 extinct peptides → AlphaFold structural features.

**What this notebook is not.** It has *not* been re-executed. Cell outputs are stripped;
every number and figure quoted in the markdown is transcribed from the original run,
which lived in Google Colab with a GPU and Google Drive mounts. See
[`archive/00_full_working_notebook.ipynb`](archive/00_full_working_notebook.ipynb)
for the unedited working notebook with its original outputs, including the abandoned
first-attempt encoder and the debugging that led to the final design.

Figures referenced below are in [`../docs/figures/`](../docs/figures/).
Full results and caveats: [`../docs/RESULTS.md`](../docs/RESULTS.md).

## 0. Configuration

All filesystem paths are set here. Adjust these for your environment; nothing else in the notebook hardcodes a path.

In [ ]:
from pathlib import Path

# Repo root, assuming this notebook is run from notebooks/
REPO_ROOT = Path("..").resolve()

# Per-peptide AlphaFold structural summaries (checked into the repo)
STRUCT_TRAIN_CSV = REPO_ROOT / "data" / "structural_features" / "train_structural_features.csv"
STRUCT_TEST_CSV  = REPO_ROOT / "data" / "structural_features" / "test_structural_features.csv"

# Held-out extinct-peptide MICs (Wan et al. 2024 supplementary — see data/external/README.md)
EXTINCT_XLSX = REPO_ROOT / "data" / "external" / "41551_2024_1201_MOESM4_ESM.xlsx"

# Where trained model weights get written
MODEL_DIR = REPO_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# Curated datasets are pulled from the HuggingFace Hub (public, no token required):
HF_REPO = "pedbb/mic_prediction"

for p in (STRUCT_TRAIN_CSV, STRUCT_TEST_CSV, EXTINCT_XLSX):
    print(f"{'OK ' if p.exists() else 'MISSING'} {p}")

## 1. Setup and data loading

`baseline.csv` is the fully curated, feature-engineered training table: one row per peptide-target MIC measurement (25,306 rows).

In [ ]:
!pip install modlamp

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from huggingface_hub import hf_hub_download
# from modlamp.descriptors import GlobalDescriptor

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    GridSearchCV
)
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.kernel_approximation import Nystroem
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import (
    LogisticRegression,
    SGDClassifier
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

In [ ]:
# RUN THIS AND NONE OF THE BELOW CELLS TO AVOID RERUNNING
file_path = hf_hub_download(
    repo_id="pedbb/mic_prediction",
    filename="baseline.csv",
    repo_type="dataset",
    token=None
)



In [ ]:
baseline_df = pd.read_csv(file_path, index_col=0)
baseline_df.head()

### Why the data is shaped this way

The raw DBAASP export is a wide matrix of peptides x 7,879 potential targets, and it is
extremely sparse — a model with 7,879 output neurons would have ~99% of them empty on any
given row. Two decisions follow:

1. **Filter to targets with >= 200 measurements**, leaving 50 bacterial targets.
2. **Melt to long format** — one row per (peptide, target) interaction, with target identity
   supplied to the model as an embedding rather than as an output dimension.

![Measurements per target](../docs/figures/01_measurements_per_target_histogram.png)
![Sparsity, 100 sampled targets](../docs/figures/02_sparsity_heatmap_100_targets.png)
![Sparsity after filtering to 50 targets](../docs/figures/03_sparsity_heatmap_filtered_50_targets.png)

The full curation and feature-engineering code (DBAASP API pull, modlAMP descriptors, k-mers)
lives in [`01_dataset_curation.ipynb`](01_dataset_curation.ipynb) and in the archive notebook.

## 2. Sequence encoder (two-arm)

Architecture follows the APEX paper: a bidirectional GRU over the tokenized peptide, plus a learned embedding for the bacterial target, concatenated into a regression head.

In [ ]:
MAX_SEQ_LENGTH = 52   # Paper uses 52
EMBEDDING_DIM = 128   # Using learned embeddings
LATENT_DIM = 128      # Hidden dimension
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-4 # Paper uses 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Tokenization
all_sequences = baseline_df['SEQUENCE'].values
unique_chars = sorted(list(set("".join(all_sequences))))
vocab = {char: idx + 1 for idx, char in enumerate(unique_chars)}
vocab['<PAD>'] = 0
vocab_size = len(vocab)

def tokenize_sequence(seq):
    return [vocab[char] for char in seq]

tokenized_sequences = [tokenize_sequence(seq) for seq in all_sequences]
X_seq_tensors = [torch.tensor(seq) for seq in tokenized_sequences]
X_seq_padded = pad_sequence(X_seq_tensors, batch_first=True, padding_value=0)

# Ensure length matches APEX spec (52)
if X_seq_padded.shape[1] > MAX_SEQ_LENGTH:
    X_seq_padded = X_seq_padded[:, :MAX_SEQ_LENGTH]
elif X_seq_padded.shape[1] < MAX_SEQ_LENGTH:
    padding = torch.zeros((X_seq_padded.shape[0], MAX_SEQ_LENGTH - X_seq_padded.shape[1]), dtype=torch.long)
    X_seq_padded = torch.cat([X_seq_padded, padding], dim=1)

print(f"Vocabulary Size: {vocab_size}")
print(f"Vocab: {vocab}")

In [ ]:
def tokenize_sequence(seq):
    # Convert string to integer list
    return [vocab[char] for char in seq]

# Tokenize all sequences
tokenized_sequences = [tokenize_sequence(seq) for seq in all_sequences]

# Pad Sequences (Convert to Tensor)
# PyTorch's pad_sequence expects a list of tensors
X_seq_tensors = [torch.tensor(seq) for seq in tokenized_sequences]
X_seq_padded = pad_sequence(X_seq_tensors, batch_first=True, padding_value=0)

# Truncate if longer than MAX_SEQ_LENGTH (though usually we pad TO max length)
# Simple truncation for safety
if X_seq_padded.shape[1] > MAX_SEQ_LENGTH:
    X_seq_padded = X_seq_padded[:, :MAX_SEQ_LENGTH]
elif X_seq_padded.shape[1] < MAX_SEQ_LENGTH:
    # Pad more if needed to hit fixed size (optional, but good for consistency)
    padding = torch.zeros((X_seq_padded.shape[0], MAX_SEQ_LENGTH - X_seq_padded.shape[1]), dtype=torch.long)
    X_seq_padded = torch.cat([X_seq_padded, padding], dim=1)

print(f"Final Input Shape: {X_seq_padded.shape}")

In [ ]:
# prepare targets
num_targets = baseline_df['TARGET ID'].max() + 1 # should be 50
X_target = torch.tensor(baseline_df['TARGET ID'].values, dtype=torch.long)
y_raw = baseline_df['MIC'].values
y_mean = y_raw.mean()
y_std = y_raw.std()
y_normalized = (y_raw - y_mean) / y_std
y = torch.tensor(y_normalized, dtype=torch.float32).unsqueeze(1)

### Target transform: log2(MIC)

MIC values span several orders of magnitude and are measured by two-fold serial dilution, so
the natural scale is log2. The first version of this model regressed on raw MIC and badly
overfit; switching to log2 and z-scoring fixed it. (That progression is preserved in the archive
notebook.)

In [ ]:
baseline_df['log_MIC'] = np.log2(baseline_df['MIC'] + 1e-6)

# Recalculate y with log M
y_raw = baseline_df['log_MIC'].values
y_mean = y_raw.mean()
y_std = y_raw.std()
y_normalized = (y_raw - y_mean) / y_std

# Update the target variable
y = torch.tensor(y_normalized, dtype=torch.float32).unsqueeze(1)

print(f"New Log-Transformed y: Mean={y_normalized.mean():.4f}, Std={y_normalized.std():.4f}")

### Model definition

In [ ]:
# model architecture
class APEXEncoder(nn.Module):
    def __init__(self, vocab_size, num_targets, embedding_dim, hidden_dim):
        super(APEXEncoder, self).__init__()

        # --- ARM 1: Peptide Encoder ---
        self.hidden_dim = hidden_dim
        self.seq_len = MAX_SEQ_LENGTH

        # 1. Input Representation
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. RNN (Paper uses GRU)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.gru_proj = nn.Linear(hidden_dim * 2, hidden_dim)

        # 3. Layer Normalization
        self.layer_norm = nn.LayerNorm(hidden_dim)

        # 4. Attention Layer 1 (Sequence-to-Sequence Interaction)
        self.W_att1 = nn.Linear(hidden_dim + embedding_dim, MAX_SEQ_LENGTH)

        # 5. Attention Layer 2 (Pooling)
        self.W_att2 = nn.Linear(hidden_dim, 1)

        # 6. Final Linear (creation of 'h')
        self.fc_final = nn.Linear(hidden_dim, hidden_dim)

        # --- ARM 2: Target Encoder ---
        self.target_embedding = nn.Embedding(num_targets, 32)

        # --- PREDICTION HEAD ---
        self.fc_out_1 = nn.Linear(hidden_dim + 32, 64)
        self.dropout = nn.Dropout(0.1)
        self.fc_out_2 = nn.Linear(64, 1)

    def forward(self, seq_input, target_input):
        # --- APEX ENCODER LOGIC ---
        x = self.embedding(seq_input)

        rnn_out, _ = self.gru(x)
        h_rnn = self.gru_proj(rnn_out)
        h_rnn = self.layer_norm(h_rnn)

        cat_features = torch.cat([h_rnn, x], dim=2)
        att1_scores = self.W_att1(cat_features)
        a1 = F.softmax(att1_scores, dim=2)

        h_att1 = torch.bmm(a1, h_rnn)

        att2_scores = self.W_att2(h_att1)
        a2 = F.softmax(att2_scores, dim=1)

        h_att2 = torch.bmm(a2.transpose(1, 2), h_att1)
        h_att2 = h_att2.squeeze(1)

        peptide_features = self.fc_final(h_att2)

        # --- MERGE WITH TARGET ---
        target_vec = self.target_embedding(target_input)
        combined = torch.cat([peptide_features, target_vec], dim=1)

        # --- OUTPUT ---
        z = F.relu(self.fc_out_1(combined))
        z = self.dropout(z)
        output = self.fc_out_2(z)

        return output

    def get_peptide_features(self, seq_input):
        # Extract purely the APEX sequence representation
        x = self.embedding(seq_input)
        rnn_out, _ = self.gru(x)
        h_rnn = self.gru_proj(rnn_out)
        h_rnn = self.layer_norm(h_rnn)

        cat_features = torch.cat([h_rnn, x], dim=2)
        a1 = F.softmax(self.W_att1(cat_features), dim=2)
        h_att1 = torch.bmm(a1, h_rnn)

        a2 = F.softmax(self.W_att2(h_att1), dim=1)
        h_att2 = torch.bmm(a2.transpose(1, 2), h_att1).squeeze(1)

        return self.fc_final(h_att2)

### Multi-seed training with early stopping

Five seeds (0, 42, 123, 2024, 999), each with its own train/validation split stratified by
target ID, max 100 epochs with early stopping on validation loss.

**Recorded result:** R2 = 0.6189 +/- 0.0139, MSE = 0.3814, Spearman rho = 0.7876

![Learning curves and parity, 5 seeds](../docs/figures/05_seq_encoder_curves_and_parity_5seeds.png)
![Parity and residuals](../docs/figures/06_seq_encoder_parity_residuals.png)

The parity plot shows the model hedging — over-predicting low MICs and under-predicting high
ones, i.e. regression toward the mean. For screening purposes rank order matters more than
absolute value, which is why Spearman rho is reported alongside R2.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

# Update the function so it stops early and graphs all
def run_experiment_smart(seed, X_seq, X_targets, y, patience=5, max_epochs=50):
    print(f"\n--- Seed {seed} ---")

    # Reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # Stratified Split
    indices = np.arange(len(y))
    try:
        # Use X_targets.numpy() if it's a tensor
        strat_labels = X_targets.numpy() if torch.is_tensor(X_targets) else X_targets
        train_idx, val_idx = train_test_split(
            indices, test_size=0.2, random_state=seed, stratify=strat_labels
        )
    except ValueError:
        train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=seed)

    # DataLoaders
    train_ds = TensorDataset(X_seq[train_idx], X_targets[train_idx], y[train_idx])
    val_ds = TensorDataset(X_seq[val_idx], X_targets[val_idx], y[val_idx])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    # Model Setup
    model = APEXEncoder(vocab_size, num_targets, EMBEDDING_DIM, LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    # Early Stopping Setup
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_weights = None

    history = {'train': [], 'val': []}

    # Training Loop
    for epoch in range(max_epochs):
        # Train
        model.train()
        train_loss = 0.0
        for s, t, label in train_loader:
            s, t, label = s.to(DEVICE), t.to(DEVICE), label.to(DEVICE)
            optimizer.zero_grad()
            out = model(s, t)
            loss = criterion(out, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * s.size(0)

        avg_train = train_loss / len(train_loader.dataset)
        history['train'].append(avg_train)

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for s, t, label in val_loader:
                s, t, label = s.to(DEVICE), t.to(DEVICE), label.to(DEVICE)
                out = model(s, t)
                loss = criterion(out, label)
                val_loss += loss.item() * s.size(0)

        avg_val = val_loss / len(val_loader.dataset)
        history['val'].append(avg_val)

        # Check Elbow
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_counter = 0
            best_model_weights = copy.deepcopy(model.state_dict()) # Save the winner
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"   Early stopping at Epoch {epoch+1} (Best Val: {best_val_loss:.4f})")
                break

    # Restore Best Weights
    model.load_state_dict(best_model_weights)

    # Final Evaluation (Collect Preds for Parity Plot)
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for s, t, label in val_loader:
            s, t = s.to(DEVICE), t.to(DEVICE)
            out = model(s, t).squeeze()
            preds.extend(out.cpu().numpy())
            actuals.extend(label.numpy())

    mse = mean_squared_error(actuals, preds)
    r2 = r2_score(actuals, preds)

    # Return history and predictions so we can plot them later
    return mse, r2, history, np.array(preds), np.array(actuals)

# RUN EXPERIMENt
seeds = [0, 42, 123, 2024, 999]
final_mses = []
final_r2s = []
all_histories = []
all_preds = []
all_actuals = []

print(f"--- Running {len(seeds)} Seeds with Early Stopping ---")

for seed in seeds:
    mse, r2, hist, preds, actuals = run_experiment_smart(
        seed, X_seq_padded, X_target, y, patience=5, max_epochs=100
    )
    final_mses.append(mse)
    final_r2s.append(r2)
    all_histories.append(hist)
    all_preds.extend(preds)
    all_actuals.extend(actuals)

print("\n" + "="*40)
print(f"Avg R2: {np.mean(final_r2s):.4f} ± {np.std(final_r2s):.4f}")
print(f"Avg MSE: {np.mean(final_mses):.4f}")
print("="*40)

# GENERATE THE GRAPHS
plt.figure(figsize=(16, 7))

# GRAPH 1: The Consensus Elbow (Learning Curve)
plt.subplot(1, 2, 1)
# Plot individual seeds faintly
for h in all_histories:
    plt.plot(h['val'], color='red', alpha=0.15)
    plt.plot(h['train'], color='blue', alpha=0.15)

# Calculate and plot Average Curve (truncating to shortest run)
min_epochs = min([len(h['val']) for h in all_histories])
avg_val_curve = np.mean([h['val'][:min_epochs] for h in all_histories], axis=0)
avg_train_curve = np.mean([h['train'][:min_epochs] for h in all_histories], axis=0)

plt.plot(avg_train_curve, label='Avg Train Loss', color='blue', linestyle='--', linewidth=2)
plt.plot(avg_val_curve, label='Avg Val Loss (The Elbow)', color='red', linewidth=3)

plt.title("Model Stability: Learning Curves", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("MSE Loss (Log2)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# GRAPH 2: The Parity Plot (Precision)
plt.subplot(1, 2, 2)
# Convert all collected predictions to numpy
all_preds = np.array(all_preds)
all_actuals = np.array(all_actuals)

plt.scatter(all_actuals, all_preds, alpha=0.2, color='darkblue', s=10, label='Test Peptides (All Seeds)')

# Perfect prediction line
min_val = min(all_actuals.min(), all_preds.min())
max_val = max(all_actuals.max(), all_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

# One-Dilution Error Margin
plt.fill_between([min_val, max_val],
                 [min_val-1, max_val-1],
                 [min_val+1, max_val+1],
                 color='red', alpha=0.1, label='±1 Dilution Step')

plt.title(f"Parity Plot (Aggregated R² = {np.mean(final_r2s):.2f})", fontsize=14)
plt.xlabel("Actual Log2 MIC", fontsize=12)
plt.ylabel("Predicted Log2 MIC", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Tree baselines

Random Forest and XGBoost regressors on the same physicochemical + one-hot-target feature set.
These were originally written as binary classifiers and rewritten as regressors so the
comparison against the encoder is like-for-like.

**Recorded result:** Neural network R2 ~ 0.62, Random Forest R2 ~ 0.61, XGBoost R2 ~ 0.43

![Baseline comparison](../docs/figures/07_baselines_rf_xgb_comparison.png)
![Baseline parity plots](../docs/figures/08_baselines_parity_plots.png)

The Random Forest essentially matches the neural network. That is the motivating observation
for the next section: the RF is *handed* engineered physicochemical descriptors, while the
sequence encoder has to infer everything from raw sequence. Giving the network the same
descriptors is the obvious next move.

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr  # <--- ADDED IMPORT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- REBUILDING BASELINES (Sandbox Mode with Graphs) ---")

# 1. CREATE SANDBOX (Deep Copy)
# This keeps your original 'baseline_df' completely safe for the Neural Net
df_tree = baseline_df.copy()

# 2. PREPARE FEATURES FOR TREES
# We remove 'TARGET ID' from the ignore list because we will One-Hot Encode it
ignore_cols = ['ID', 'SEQUENCE', 'MIC', 'is_AMP', 'log_MIC', 'TARGET ID']
cols_to_drop = [c for c in ignore_cols if c in df_tree.columns]

X_tree = df_tree.drop(columns=cols_to_drop)
y_tree = df_tree['log_MIC']

# 3. ONE-HOT ENCODE TARGETS (Crucial for Fairness)
# This gives the Trees the same "Species Info" that the Neural Net has
if 'TARGET ID' in df_tree.columns:
    dummies = pd.get_dummies(df_tree['TARGET ID'], prefix='Target')
    X_tree = pd.concat([X_tree, dummies], axis=1)

print(f"Tree Feature Count: {X_tree.shape[1]} (Safe from NN variables)")

# 4. EVALUATION & PLOTTING FUNCTION
def evaluate_baseline_with_graphs(model_name, model_obj, X, y, seeds=[0, 42, 123, 2024, 999]):
    print(f"\n{'='*40}")
    print(f"Testing {model_name}...")
    print(f"{'='*40}")

    r2_scores = []
    mse_scores = []
    rho_scores = []  # <--- NEW LIST
    all_preds = []
    all_actuals = []

    # The Loop (Same as NN)
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed
        )

        model_obj.random_state = seed
        model_obj.fit(X_train, y_train)

        preds = model_obj.predict(X_test)

        # Metrics
        r2_scores.append(r2_score(y_test, preds))
        mse_scores.append(mean_squared_error(y_test, preds))

        # Spearman Calculation
        rho, _ = spearmanr(y_test, preds)
        rho_scores.append(rho)

        # Collect data for big graph
        all_preds.extend(preds)
        all_actuals.extend(y_test)

    # Stats
    avg_r2 = np.mean(r2_scores)
    std_r2 = np.std(r2_scores)
    avg_rho = np.mean(rho_scores)  # <--- NEW STAT
    std_rho = np.std(rho_scores)   # <--- NEW STAT

    print(f"Avg R2:       {avg_r2:.4f} ± {std_r2:.4f}")
    print(f"Avg Spearman: {avg_rho:.4f} ± {std_rho:.4f}")
    print(f"Avg MSE:      {np.mean(mse_scores):.4f}")

    # --- PLOTTING (Matching your NN Graphs) ---
    all_preds = np.array(all_preds)
    all_actuals = np.array(all_actuals)

    plt.figure(figsize=(12, 5))

    # Plot 1: Parity Plot (Accuracy)
    plt.subplot(1, 2, 1)
    plt.scatter(all_actuals, all_preds, alpha=0.1, color='green', s=10)
    # Perfect Line
    min_val, max_val = min(all_actuals.min(), all_preds.min()), max(all_actuals.max(), all_preds.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect')
    # +/- 1 Dilution Error
    plt.fill_between([min_val, max_val], [min_val-1, max_val-1], [min_val+1, max_val+1], color='red', alpha=0.1)

    plt.title(f"{model_name}: Parity Plot\nR²={avg_r2:.2f} | Rho={avg_rho:.2f}")
    plt.xlabel("Actual Log2 MIC")
    plt.ylabel("Predicted Log2 MIC")
    plt.grid(True, alpha=0.3)

    # Plot 2: Residual Plot (Slope Check)
    plt.subplot(1, 2, 2)
    residuals = all_preds - all_actuals
    plt.scatter(all_actuals, residuals, alpha=0.1, color='purple', s=10)
    plt.axhline(0, color='black', linestyle='--')
    plt.title(f"{model_name}: Residuals")
    plt.xlabel("Actual Log2 MIC")
    plt.ylabel("Error")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return avg_r2, avg_rho

# 5. RUN THE SHOWDOWN
# XGBoost (Fast, Gradient Boosting)
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, n_jobs=-1)
xgb_r2, xgb_rho = evaluate_baseline_with_graphs("XGBoost", xgb, X_tree, y_tree)

# Random Forest (Classic Bagging)
rf = RandomForestRegressor(n_estimators=100, max_depth=20, n_jobs=-1)
rf_r2, rf_rho = evaluate_baseline_with_graphs("Random Forest", rf, X_tree, y_tree)

print("\n" + "="*40)
print("FINAL 1D LEADERBOARD (R² / Spearman)")
print("="*40)
print(f"1. XGBoost:       R² ≈ {xgb_r2:.2f} | Rho ≈ {xgb_rho:.2f}")
print(f"2. Random Forest: R² ≈ {rf_r2:.2f} | Rho ≈ {rf_rho:.2f}")
# Add your NN score manually if you know it, e.g.:
# print(f"3. Neural Network: R² ≈ 0.62 | Rho ≈ 0.65 (Confirmed)")

## 4. Hybrid model (three-arm)

Adds a third branch: 11 standardized physicochemical descriptors fed through a dense block and concatenated with the sequence and target representations.

In [ ]:
from sklearn.preprocessing import StandardScaler
import torch
import pandas as pd
import numpy as np

print("--- PREPARING PHYSICOCHEMICAL FEATURES (Branch 3) ---")

# 1. Select the Columns (Same concept as the Random Forest)
# We exclude ID, Sequence, and Targets (since Targets have their own branch)
ignore_cols = ['ID', 'SEQUENCE', 'MIC', 'is_AMP', 'log_MIC', 'TARGET ID']
cols_to_drop = [c for c in ignore_cols if c in baseline_df.columns]

# Create a clean DataFrame of just the numbers (Charge, MW, etc.)
# We use .copy() to ensure we don't touch the original df
df_features = baseline_df.drop(columns=cols_to_drop).copy()

# 2. Normalize (CRITICAL for Neural Networks)
# Random Forests didn't need this, but NNs will explode without it.
scaler = StandardScaler()
X_phys_array = scaler.fit_transform(df_features)

# 3. Convert to Tensor
X_phys = torch.tensor(X_phys_array, dtype=torch.float32)
num_phys_features = X_phys.shape[1]

print(f"Original DataFrame Shape: {baseline_df.shape} (Untouched)")
print(f"New Phys Feature Tensor:  {X_phys.shape}")
print(f"Number of Phys Features:  {num_phys_features}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class APEXHybrid(nn.Module):
    def __init__(self, vocab_size, num_targets, num_phys_features, embedding_dim=128, hidden_dim=128):
        super(APEXHybrid, self).__init__()

        # --- BRANCH 1: Sequence (The "Deep" Part) ---
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.gru_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        # Attention
        self.W_att1 = nn.Linear(hidden_dim + embedding_dim, 52)
        self.W_att2 = nn.Linear(hidden_dim, 1)
        self.fc_seq = nn.Linear(hidden_dim, hidden_dim)

        # --- BRANCH 2: Target (The Context Part) ---
        self.target_embedding = nn.Embedding(num_targets, 32)

        # --- BRANCH 3: Physicochemical (The "Wide" Part) ---
        # We process the raw features slightly before merging
        # Input: num_phys_features -> Output: 32 dimensions
        self.phys_layer = nn.Linear(num_phys_features, 32)

        # --- MERGE & OUTPUT ---
        # Total Size = Sequence(128) + Target(32) + Phys(32)
        total_dim = hidden_dim + 32 + 32

        self.fc_1 = nn.Linear(total_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Linear(64, 1)

    def forward(self, seq_input, target_input, phys_input):
        # 1. Sequence Branch
        x = self.embedding(seq_input)
        rnn_out, _ = self.gru(x)
        h_rnn = self.layer_norm(self.gru_proj(rnn_out))

        # Attention
        cat_features = torch.cat([h_rnn, x], dim=2)
        a1 = F.softmax(self.W_att1(cat_features), dim=2)
        h_att1 = torch.bmm(a1, h_rnn)
        a2 = F.softmax(self.W_att2(h_att1), dim=1)
        h_att2 = torch.bmm(a2.transpose(1, 2), h_att1).squeeze(1)
        seq_vec = self.fc_seq(h_att2)

        # 2. Target Branch
        target_vec = self.target_embedding(target_input)

        # 3. Phys Branch (New!)
        phys_vec = F.relu(self.phys_layer(phys_input))

        # 4. Combine
        combined = torch.cat([seq_vec, target_vec, phys_vec], dim=1)

        z = F.relu(self.fc_1(combined))
        z = self.dropout(z)
        output = self.fc_out(z)

        return output

**Recorded result:** R2 = 0.6295 +/- 0.0239, MSE = 0.3706

![Hybrid training curves](../docs/figures/09_hybrid_3arm_training_curves.png)

A marginal gain over the two-arm encoder (0.6295 vs 0.6189) and over the Random Forest (~0.61).
Linear models were also checked and are far worse (Nystrom-SGD R2 = 0.149, Linear SVM R2 = 0.169),
which is expected — the structure-activity relationship is not linear.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

# --- 1. THE HYBRID FUNCTION (Accepts X_phys) ---
def run_hybrid_experiment_smart(seed, X_seq, X_targets, X_phys, y, patience=5, max_epochs=100):
    print(f"\n--- Hybrid Seed {seed} ---")

    # Reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # Stratified Split
    indices = np.arange(len(y))
    try:
        strat_labels = X_targets.numpy() if torch.is_tensor(X_targets) else X_targets
        train_idx, val_idx = train_test_split(
            indices, test_size=0.2, random_state=seed, stratify=strat_labels
        )
    except ValueError:
        train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=seed)

    # DataLoaders (NOW WITH 4 TENSORS: Seq, Target, Phys, Label)
    train_ds = TensorDataset(X_seq[train_idx], X_targets[train_idx], X_phys[train_idx], y[train_idx])
    val_ds = TensorDataset(X_seq[val_idx], X_targets[val_idx], X_phys[val_idx], y[val_idx])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    # Model Setup (APEXHybrid)
    model = APEXHybrid(vocab_size, num_targets, num_phys_features, EMBEDDING_DIM, LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    # Early Stopping Setup
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_weights = None

    history = {'train': [], 'val': []}

    # --- Training Loop ---
    for epoch in range(max_epochs):
        # A. Train
        model.train()
        train_loss = 0.0
        for s, t, p, label in train_loader:
            s, t, p, label = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE), label.to(DEVICE)
            optimizer.zero_grad()
            out = model(s, t, p) # Pass 3 inputs
            loss = criterion(out, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * s.size(0)

        avg_train = train_loss / len(train_loader.dataset)
        history['train'].append(avg_train)

        # B. Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for s, t, p, label in val_loader:
                s, t, p, label = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE), label.to(DEVICE)
                out = model(s, t, p) # Pass 3 inputs
                loss = criterion(out, label)
                val_loss += loss.item() * s.size(0)

        avg_val = val_loss / len(val_loader.dataset)
        history['val'].append(avg_val)

        # C. Check Elbow
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_counter = 0
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"   Early stopping at Epoch {epoch+1} (Best Val: {best_val_loss:.4f})")
                break

    # Restore Best Weights
    model.load_state_dict(best_model_weights)

    # Final Evaluation
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for s, t, p, label in val_loader:
            s, t, p = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE)
            out = model(s, t, p).squeeze()
            preds.extend(out.cpu().numpy())
            actuals.extend(label.numpy())

    mse = mean_squared_error(actuals, preds)
    r2 = r2_score(actuals, preds)

    return mse, r2, history, np.array(preds), np.array(actuals)

# --- 2. RUN THE EXPERIMENT ---
seeds = [0, 42, 123, 2024, 999]
final_mses = []
final_r2s = []
all_histories = []
all_preds = []
all_actuals = []

print(f"--- Running Hybrid Model (5 Seeds) ---")

for seed in seeds:
    mse, r2, hist, preds, actuals = run_hybrid_experiment_smart(
        seed, X_seq_padded, X_target, X_phys, y, patience=5, max_epochs=100
    )
    final_mses.append(mse)
    final_r2s.append(r2)
    all_histories.append(hist)
    all_preds.extend(preds)
    all_actuals.extend(actuals)

print("\n" + "="*40)
print(f"Avg R2: {np.mean(final_r2s):.4f} ± {np.std(final_r2s):.4f}")
print(f"Avg MSE: {np.mean(final_mses):.4f}")
print("="*40)

# --- 3. GENERATE THE GRAPHS (Identical to 2-Branch for Comparison) ---
plt.figure(figsize=(16, 7))

# GRAPH 1: Learning Curve
plt.subplot(1, 2, 1)
for h in all_histories:
    plt.plot(h['val'], color='red', alpha=0.15)
    plt.plot(h['train'], color='blue', alpha=0.15)

min_epochs = min([len(h['val']) for h in all_histories])
avg_val_curve = np.mean([h['val'][:min_epochs] for h in all_histories], axis=0)
avg_train_curve = np.mean([h['train'][:min_epochs] for h in all_histories], axis=0)

plt.plot(avg_train_curve, label='Avg Train Loss', color='blue', linestyle='--', linewidth=2)
plt.plot(avg_val_curve, label='Avg Val Loss (The Elbow)', color='red', linewidth=3)
plt.title("Hybrid Model: Learning Curves", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("MSE Loss (Log2)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# GRAPH 2: Parity Plot
plt.subplot(1, 2, 2)
all_preds = np.array(all_preds)
all_actuals = np.array(all_actuals)

plt.scatter(all_actuals, all_preds, alpha=0.2, color='darkgreen', s=10, label='Hybrid Predictions')

min_val = min(all_actuals.min(), all_preds.min())
max_val = max(all_actuals.max(), all_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.fill_between([min_val, max_val],
                 [min_val-1, max_val-1],
                 [min_val+1, max_val+1],
                 color='red', alpha=0.1, label='±1 Dilution Step')

plt.title(f"Hybrid Parity Plot (Aggregated R² = {np.mean(final_r2s):.2f})", fontsize=14)
plt.xlabel("Actual Log2 MIC", fontsize=12)
plt.ylabel("Predicted Log2 MIC", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Ensemble

Averaging the five per-seed hybrid models.

**Recorded result:** R2 = 0.6257 +/- 0.0095, MSE = 0.3741

![Ensemble curves and parity](../docs/figures/10_ensemble_curves_and_parity.png)

The ensemble mean is *slightly below* the best single model but has roughly half the variance
across seeds (+/- 0.0095 vs +/- 0.0239). The ensemble was used for the final evaluation on that basis.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import copy

# --- CONFIGURATION ---
ENSEMBLE_SIZE = 5      # Number of models inside ONE ensemble
MAX_EPOCHS = 30        # Epochs per model
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seeds = [0, 42, 123, 2024, 999] # The rigorous outer loop

# Helper to get your specific model
def get_fresh_model():
    return APEXHybrid(
        vocab_size=vocab_size,
        num_targets=num_targets,
        num_phys_features=num_phys_features,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=LATENT_DIM
    ).to(DEVICE)

# --- 1. THE ENSEMBLE EXPERIMENT FUNCTION ---
def run_ensemble_experiment(seed, X_seq, X_targets, X_phys, y):
    print(f"\n=== Experiment Seed {seed} ===")

    # A. Reproducible Split
    torch.manual_seed(seed)
    np.random.seed(seed)
    indices = np.arange(len(y))

    # Stratified split if possible
    try:
        strat_labels = X_targets.numpy() if torch.is_tensor(X_targets) else X_targets
        train_idx, val_idx = train_test_split(indices, test_size=0.1, random_state=seed, stratify=strat_labels)
    except:
        train_idx, val_idx = train_test_split(indices, test_size=0.1, random_state=seed)

    # B. DataLoaders
    train_ds = TensorDataset(X_seq[train_idx], X_targets[train_idx], X_phys[train_idx], y[train_idx])
    val_ds   = TensorDataset(X_seq[val_idx],   X_targets[val_idx],   X_phys[val_idx],   y[val_idx])

    # Persistent val_loader for evaluation
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # Store individual model predictions to average later
    member_preds = []

    # Store average history across members to plot "Ensemble Stability"
    ensemble_train_losses = np.zeros(MAX_EPOCHS)
    ensemble_val_losses = np.zeros(MAX_EPOCHS)

    # C. Train the Ensemble Members
    for i in range(ENSEMBLE_SIZE):
        # Sub-seed for diversity within the ensemble
        sub_seed = seed + (i * 100)
        torch.manual_seed(sub_seed)

        model = get_fresh_model()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.MSELoss()

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

        print(f"   -> Training Member {i+1}/{ENSEMBLE_SIZE}...")

        for epoch in range(MAX_EPOCHS):
            # Train
            model.train()
            train_loss = 0.0
            for s, t, p, label in train_loader:
                s, t, p, label = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE), label.to(DEVICE)
                optimizer.zero_grad()
                out = model(s, t, p)
                loss = criterion(out, label)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * s.size(0)

            # Validation (for history)
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for s, t, p, label in val_loader:
                    s, t, p, label = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE), label.to(DEVICE)
                    out = model(s, t, p)
                    l = criterion(out, label)
                    val_loss += l.item() * s.size(0)

            # Accumulate for average history
            ensemble_train_losses[epoch] += (train_loss / len(train_ds))
            ensemble_val_losses[epoch] += (val_loss / len(val_ds))

        # Member Inference
        model.eval()
        preds = []
        with torch.no_grad():
            for s, t, p, label in val_loader:
                s, t, p = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE)
                out = model(s, t, p)
                preds.extend(out.cpu().numpy().flatten())
        member_preds.append(preds)

    # D. Aggregate Results
    # 1. Average History
    avg_history = {
        'train': ensemble_train_losses / ENSEMBLE_SIZE,
        'val': ensemble_val_losses / ENSEMBLE_SIZE
    }

    # 2. Ensemble Prediction (Mean of members)
    ensemble_final_preds = np.mean(member_preds, axis=0)

    # 3. Get Actuals
    actuals = y[val_idx].cpu().numpy().flatten()

    # 4. Metrics
    mse = mean_squared_error(actuals, ensemble_final_preds)
    r2 = r2_score(actuals, ensemble_final_preds)

    return mse, r2, avg_history, ensemble_final_preds, actuals

# --- 2. RUN THE OUTER LOOP ---
final_mses = []
final_r2s = []
all_histories = []
all_preds = []
all_actuals = []

print(f"Starting Robust Ensemble Evaluation ({len(seeds)} Seeds x {ENSEMBLE_SIZE} Models)...")

for seed in seeds:
    mse, r2, hist, preds, actuals = run_ensemble_experiment(
        seed, X_seq_padded, X_target, X_phys, y
    )
    final_mses.append(mse)
    final_r2s.append(r2)
    all_histories.append(hist)
    all_preds.extend(preds)
    all_actuals.extend(actuals)

print("\n" + "="*40)
print(f"Overall Ensemble R2: {np.mean(final_r2s):.4f} ± {np.std(final_r2s):.4f}")
print(f"Overall Ensemble MSE: {np.mean(final_mses):.4f}")
print("="*40)

# --- 3. GENERATE THE EXACT SAME GRAPHS ---
plt.figure(figsize=(16, 7))

# GRAPH 1: Average Member Stability (Learning Curve)
plt.subplot(1, 2, 1)
# Plot individual seed averages faintly
for h in all_histories:
    plt.plot(h['val'], color='red', alpha=0.15)
    plt.plot(h['train'], color='blue', alpha=0.15)

# Grand Average across all seeds
avg_val_curve = np.mean([h['val'] for h in all_histories], axis=0)
avg_train_curve = np.mean([h['train'] for h in all_histories], axis=0)

plt.plot(avg_train_curve, label='Avg Member Train Loss', color='blue', linestyle='--', linewidth=2)
plt.plot(avg_val_curve, label='Avg Member Val Loss', color='red', linewidth=3)

plt.title(f"Ensemble Stability (Avg of {len(seeds)} Seeds)", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("MSE Loss (Log2)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# GRAPH 2: Parity Plot
plt.subplot(1, 2, 2)
all_preds = np.array(all_preds)
all_actuals = np.array(all_actuals)

plt.scatter(all_actuals, all_preds, alpha=0.2, color='purple', s=10, label='Ensemble Predictions')

# Perfect prediction line
min_val = min(all_actuals.min(), all_preds.min())
max_val = max(all_actuals.max(), all_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

# One-Dilution Error Margin
plt.fill_between([min_val, max_val],
                 [min_val-1, max_val-1],
                 [min_val+1, max_val+1],
                 color='red', alpha=0.1, label='±1 Dilution Step')

plt.title(f"Ensemble Parity Plot (Aggregated R² = {np.mean(final_r2s):.2f})", fontsize=14)
plt.xlabel("Actual Log2 MIC", fontsize=12)
plt.ylabel("Predicted Log2 MIC", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Out-of-distribution evaluation: 69 extinct peptides

The real test. All results above are in-distribution — modern peptides from DBAASP, split
randomly. Here the models are applied to 69 peptides computationally resurrected from extinct
organisms (Wan et al. 2024), against 11 bacterial strains, giving 690 peptide-target pairs.

The steps are: recover the exact target-name to target-ID mapping used at training time, load
and melt the supplementary MIC table, recompute modlAMP descriptors on the extinct sequences,
and align them to the same 11 features. `SYNTHESIS TYPE` is dropped because it is unavailable
for the extinct set.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

# 1. Download Master Data to relearn the IDs
print("Fetching master data...")
file_path = hf_hub_download(repo_id="pedbb/mic_prediction", filename="master_peptide_data.csv", repo_type="dataset")
df_master = pd.read_csv(file_path)

# 2. Sort targets exactly like training (Most data -> ID 0)
sequence_info = ['COMPLEXITY', 'NAME', 'N TERMINUS', 'SEQUENCE', 'C TERMINUS', 'SYNTHESIS TYPE', 'TARGET GROUP', 'TARGET OBJECT', 'Molecular_Weight']
mic_df = df_master.drop(columns=sequence_info).reset_index(drop=True)
nulls = mic_df.shape[0] - mic_df.drop(columns=['ID']).isnull().sum()
targets = nulls[nulls >= 200].sort_values(ascending=False).index.to_list()

# 3. Create the Map
target_map = {name: i for i, name in enumerate(targets)}
print(f"✅ ID Map Recovered! (ID 0 = {targets[0]})")

# Add this to verify
print("Top 5 Targets:", targets[:5])
# Expected: ['Escherichia coli', 'Staphylococcus aureus', ...]

In [ ]:
import pandas as pd
import numpy as np

# 1. CONFIGURATION
# Pointing directly to your Excel file in Drive
TEST_FILE = EXTINCT_XLSX

# 2. LOAD & MELT
print(f"Loading {TEST_FILE}...")
# header=1 skips the first "garbage" row so we get the real column names
df_wide = pd.read_excel(TEST_FILE, header=1)

# List of the 11 bacteria columns (Exact names from the file)
bacteria_cols = [
    'A. baumannii ATCC19606', 'E. coli ATCC11775', 'E. coli AIG221', 'E. coli AIG222',
    'K. pneumoniae ATCC13883', 'P. aeruginosa PAO1', 'P. aeruginosa PA14',
    'S. aureus ATCC12600', 'S. aureus (ATCC BAA-1556) - MRSA',
    'vancomycin-resistant E. faecalis ATCC700802', 'vancomycin-resistant E. faecium ATCC700221'
]

# Melt into long format (creating one row per bacteria-peptide pair)
df_long = df_wide.melt(id_vars=['Seq'], value_vars=[c for c in bacteria_cols if c in df_wide.columns],
                       var_name='Bacteria_Name', value_name='True_MIC')

# 3. CLEANING
# Convert N.A. to 128 (Inactive) and clean sequences
df_long['True_MIC'] = pd.to_numeric(
    df_long['True_MIC'].astype(str).replace(['N.A.', 'nan', '>', '<'], ['128', '128', '', ''], regex=True),
    errors='coerce'
).fillna(128)

# Remove non-standard characters and filter valid peptides
df_long['Clean_Seq'] = df_long['Seq'].str.upper().str.replace('[^A-Z]', '', regex=True)
df_long = df_long[df_long['Clean_Seq'].str.contains('^[ACDEFGHIKLMNPQRSTVWY]+$')].reset_index(drop=True)

# 4. ROBUST MAPPING LOGIC
def get_id(name):
    # Step A: Remove the prefix that breaks the split logic
    clean_name = name.replace("vancomycin-resistant ", "")

    parts = clean_name.split(' ')
    if len(parts) < 2: return -1

    genus_abbr = parts[0]
    species = parts[1]

    # Step B: Biologically accurate expansion
    if genus_abbr == 'E.':
        # "E." can be Escherichia OR Enterococcus
        if species.startswith('faec'): genus = 'Enterococcus'
        else: genus = 'Escherichia'
    elif genus_abbr == 'P.': genus = 'Pseudomonas'
    elif genus_abbr == 'S.': genus = 'Staphylococcus'
    elif genus_abbr == 'K.': genus = 'Klebsiella'
    elif genus_abbr == 'A.': genus = 'Acinetobacter'
    else: genus = genus_abbr

    full_name = f"{genus} {species}"

    # Step C: Fuzzy Match against your Master Target Map
    for k, v in target_map.items():
        if full_name in k:
            return v

    print(f"⚠️ Could not map: {name} -> {full_name}")
    return -1

# Apply the map
df_long['TARGET_ID'] = df_long['Bacteria_Name'].apply(get_id)

# Filter out anything that didn't match (-1)
df_final_test = df_long[df_long['TARGET_ID'] != -1].copy().reset_index(drop=True)

# 5. EXPORT SEQUENCE LIST
sequences_69 = df_final_test['Clean_Seq'].unique().tolist()
print(f"✅ Ready! {len(df_final_test)} measurements mapped.")
print(f"   (Derived from {len(sequences_69)} unique sequences)")
print(df_final_test[['Bacteria_Name', 'TARGET_ID']].drop_duplicates())

In [ ]:
!pip install modlamp

In [ ]:
from modlamp.descriptors import GlobalDescriptor
import pandas as pd
import numpy as np

print(f"Calculating features for {len(sequences_69)} unique sequences...")

# 1. Initialize & Calculate Length
g = GlobalDescriptor(sequences_69)
g.length() # This initializes the matrix and calculates Feature 0 (Length)

# 2. Calculate the rest of the standard descriptors
g.calculate_MW(append=True)             # 1. Mol Weight
g.calculate_charge(append=True)         # 2. Charge
g.isoelectric_point(append=True)        # 3. Isoelectric Point
g.instability_index(append=True)        # 4. Instability
g.aromaticity(append=True)              # 5. Aromaticity
g.aliphatic_index(append=True)          # 6. Aliphatic
g.boman_index(append=True)              # 7. Boman
g.hydrophobic_ratio(append=True)        # 8. Hydrophobicity

# 3. Calculate Charge Density (separate object)
cd = GlobalDescriptor(sequences_69)
cd.charge_density()

# 4. Calculate COMPLEXITY (The feature your training set has but ModLAMP doesn't)
# defined as: Unique AAs / Total Length
complexity = [len(set(s)) / len(s) for s in sequences_69]

# 5. Assemble into a DataFrame with EXACT Training Column Names
# Training Order: [COMPLEXITY, LENGTH, MW, CHARGE, CHARGE_DENSITY, ISOELECTRIC, INSTABILITY, AROMATICITY, ALIPHATIC, BOMAN, HYDROPHOBICITY]

df_features_69 = pd.DataFrame()
df_features_69['SEQUENCE'] = sequences_69 # Keep track of sequence
df_features_69['COMPLEXITY'] = complexity

# Map ModLAMP outputs to columns
# g.descriptor columns: [0:Len, 1:MW, 2:Chg, 3:pI, 4:Inst, 5:Arom, 6:Aliph, 7:Boman, 8:Hydro]
df_features_69['SEQUENCE LENGTH'] = g.descriptor[:, 0]
df_features_69['MOLECULAR WEIGHT'] = g.descriptor[:, 1]
df_features_69['CHARGE'] = g.descriptor[:, 2]
df_features_69['CHARGE DENSITY'] = cd.descriptor.flatten() # Insert Charge Density here
df_features_69['ISOELECTRIC POINT'] = g.descriptor[:, 3]
df_features_69['INSTABILITY INDEX'] = g.descriptor[:, 4]
df_features_69['AROMATICITY'] = g.descriptor[:, 5]
df_features_69['ALIPHATIC INDEX'] = g.descriptor[:, 6]
df_features_69['BOWMAN INDEX'] = g.descriptor[:, 7]
df_features_69['HYDROPHOBIC RATIO'] = g.descriptor[:, 8]

# 6. Final Feature Matrix (X_phys)
# Drop the SEQUENCE column to get just the numbers
X_phys_extinct = df_features_69.drop(columns=['SEQUENCE']).values

print(f"✅ Features Generated: {X_phys_extinct.shape}")
print("Columns:", df_features_69.drop(columns=['SEQUENCE']).columns.tolist())

In [ ]:
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np

# 1. Define the Exact 11 Features (Matching your Extinct Data generator)
# We explicitly exclude 'SYNTHESIS TYPE' from this list.
train_feature_cols = [
    'COMPLEXITY',
    'SEQUENCE LENGTH', 'MOLECULAR WEIGHT', 'CHARGE', 'CHARGE DENSITY',
    'ISOELECTRIC POINT', 'INSTABILITY INDEX', 'AROMATICITY',
    'ALIPHATIC INDEX', 'BOWMAN INDEX', 'HYDROPHOBIC RATIO'
]

print(f"Selected {len(train_feature_cols)} features for training.")

# 2. Create the Scaler & Transform (New Variable)
# We fit the scaler ONLY on these 11 columns.
scaler = StandardScaler()

try:
    # We verify the columns exist first
    missing = [c for c in train_feature_cols if c not in baseline_df.columns]
    if missing:
        raise KeyError(f"baseline_df is missing: {missing}")

    # Create the new tensor (X_phys) without modifying baseline_df
    X_phys_array = scaler.fit_transform(baseline_df[train_feature_cols].values)
    X_phys = torch.tensor(X_phys_array, dtype=torch.float32)

    print(f"✅ Training Physics Tensor Created.")
    print(f"   Shape: {X_phys.shape} (Rows x 11)")
    print(f"   Status: 'SYNTHESIS TYPE' successfully ignored.")

except Exception as e:
    print(f"❌ Error preparing features: {e}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

print("--- PREPARING HYBRID TEST LOADER (11 FEATURES) ---")

# 1. SETUP FEATURES (11 Total)
# We use the 10 ModLAMP features PLUS Complexity.
# We explicitly exclude 'SYNTHESIS TYPE'.
phys_cols = [
    'COMPLEXITY',
    'SEQUENCE LENGTH', 'MOLECULAR WEIGHT', 'CHARGE', 'CHARGE DENSITY',
    'ISOELECTRIC POINT', 'INSTABILITY INDEX', 'AROMATICITY',
    'ALIPHATIC INDEX', 'BOWMAN INDEX', 'HYDROPHOBIC RATIO'
]

print(f"Features selected ({len(phys_cols)}): {phys_cols}")

# A. Fit Scaler on Baseline Data (Training Stats)
# This ensures we scale the mammoth peptides using the same logic as the training set
scaler = StandardScaler()
try:
    # Check if baseline_df has all columns
    missing = [c for c in phys_cols if c not in baseline_df.columns]
    if missing:
        raise KeyError(f"baseline_df is missing columns: {missing}")

    scaler.fit(baseline_df[phys_cols].values)
    print("✅ Scaler fitted on baseline_df.")

except Exception as e:
    print(f"❌ Error fitting scaler: {e}")
    raise e

# B. Prepare Extinct Physics Features
# Merge the features you generated (df_features_69) into the mapped test set (df_final_test)
df_test_ready = pd.merge(
    df_final_test,
    df_features_69,
    left_on='Clean_Seq',
    right_on='SEQUENCE',
    how='left'
)

# C. Scale the Test Data
# Transform using the scaler trained on baseline_df
try:
    X_phys_test_array = scaler.transform(df_test_ready[phys_cols].values)
    X_phys_test = torch.tensor(X_phys_test_array, dtype=torch.float32)
except Exception as e:
    print(f"❌ Error transforming test data: {e}")
    # Debug: Check what columns df_test_ready actually has
    print("Available columns:", df_test_ready.columns.tolist())
    raise e

# 2. SETUP SEQUENCES (Tokenization)
aa_vocab = {k: v for v, k in enumerate("ACDEFGHIKLMNPQRSTVWY", start=1)}
max_len = 52 # Matches original paper & training logic

def tokenize(seq, vocab, max_l):
    indices = [vocab.get(c, 0) for c in seq]
    if len(indices) < max_l:
        indices += [0] * (max_l - len(indices))
    else:
        indices = indices[:max_l]
    return indices

print(f"Tokenizing sequences (Max Len: {max_len})...")
X_seq_list = [tokenize(s, aa_vocab, max_len) for s in df_test_ready['Clean_Seq']]
X_seq_test = np.array(X_seq_list)

# 3. CREATE TENSORS (Three Arms)
tensor_seq = torch.tensor(X_seq_test, dtype=torch.long)
tensor_target = torch.tensor(df_test_ready['TARGET_ID'].values, dtype=torch.long)
tensor_phys = X_phys_test # Already a tensor from step C
tensor_y_true = torch.tensor(df_test_ready['True_MIC'].values, dtype=torch.float32)

# 4. DATALOADER
test_dataset = TensorDataset(tensor_seq, tensor_target, tensor_phys, tensor_y_true)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✅ Ready for Hybrid Inference!")
print(f"   Samples: {len(test_dataset)}")
print(f"   Inputs: Seq={tensor_seq.shape}, Target={tensor_target.shape}, Phys={tensor_phys.shape}")

### First attempt, and the scaling bug

The first OOD run produced implausible MIC predictions — the model had been trained on
z-scored log2 targets, but the extinct targets were not put on the same scale.

![Unscaled extinct predictions](../docs/figures/11_extinct_predictions_unscaled.png)

**Before fix:** ensemble Spearman rho = 0.3025
**After applying the identical log2 + z-score transform:** ensemble Spearman rho = 0.3564

![Rescaled extinct predictions: 3-seed ensemble parity, rho = 0.356](../docs/figures/12_extinct_ensemble_parity_zscored.png)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import copy
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CONFIGURATION ---
MAX_LEN = 52
SEEDS = [42, 123, 2024]
SAVE_DIR = MODEL_DIR
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"--- 1. DATA PREP (11 Features | Log2 + Z-Score Targets) ---")

# A. FEATURES
phys_cols_11 = [
    'COMPLEXITY',
    'SEQUENCE LENGTH', 'MOLECULAR WEIGHT', 'CHARGE', 'CHARGE DENSITY',
    'ISOELECTRIC POINT', 'INSTABILITY INDEX', 'AROMATICITY',
    'ALIPHATIC INDEX', 'BOWMAN INDEX', 'HYDROPHOBIC RATIO'
]

# B. PREPARE TRAINING DATA
# 1. Physics
scaler = StandardScaler()
# Add complexity if missing
if 'COMPLEXITY' not in baseline_df.columns:
     baseline_df['COMPLEXITY'] = [len(set(s)) / len(s) for s in baseline_df['SEQUENCE']]
X_phys_train_array = scaler.fit_transform(baseline_df[phys_cols_11].values)
X_phys_temp = torch.tensor(X_phys_train_array, dtype=torch.float32)

# 2. Sequences
if 'aa_vocab' not in locals():
    aa_vocab = {k: v for v, k in enumerate("ACDEFGHIKLMNPQRSTVWY", start=1)}

def tokenize(seq, vocab, max_l):
    indices = [vocab.get(c, 0) for c in seq]
    if len(indices) < max_l:
        indices += [0] * (max_l - len(indices))
    else:
        indices = indices[:max_l]
    return indices

train_seq_list = [tokenize(s, aa_vocab, MAX_LEN) for s in baseline_df['SEQUENCE']]
X_seq_temp = torch.tensor(train_seq_list, dtype=torch.long)

# 3. Targets
if 'TARGET ID' not in baseline_df.columns:
    raise KeyError("Column 'TARGET ID' missing from baseline_df")
X_targets_temp = torch.tensor(baseline_df['TARGET ID'].values, dtype=torch.long)

# 4. Labels (Log2 AND Z-Score)
# First, get Log2
if 'LOG2_MIC' in baseline_df.columns:
    y_log2 = baseline_df['LOG2_MIC'].values
else:
    y_log2 = np.log2(baseline_df['MIC'].values + 1e-6)

# CALCULATE TRAINING STATISTICS (Critical Step)
Y_MEAN = y_log2.mean()
Y_STD = y_log2.std()
print(f"   Training Target Stats: Mean={Y_MEAN:.4f}, Std={Y_STD:.4f}")

# Z-Score Normalize Training Targets
y_normalized = (y_log2 - Y_MEAN) / Y_STD
y_temp = torch.tensor(y_normalized, dtype=torch.float32)

# C. SANITIZE
max_id = len(target_map) - 1
valid_mask = (X_targets_temp >= 0) & (X_targets_temp <= max_id)

X_seq_padded = X_seq_temp[valid_mask]
X_targets = X_targets_temp[valid_mask]
X_phys = X_phys_temp[valid_mask]
y = y_temp[valid_mask]

print(f"✅ Training Data: {len(y)} samples (Z-Scored)")

# D. PREPARE EXTINCT DATA
# 1. Sync Features
df_extinct = df_test_ready.copy()
if 'COMPLEXITY' not in df_extinct.columns:
    df_extinct['COMPLEXITY'] = [len(set(s)) / len(s) for s in df_extinct['Clean_Seq']]

# 2. Scale Physics
X_phys_ext_array = scaler.transform(df_extinct[phys_cols_11].values)
tensor_phys_ext = torch.tensor(X_phys_ext_array, dtype=torch.float32)

# 3. Sequence & Target
ext_seq_list = [tokenize(s, aa_vocab, MAX_LEN) for s in df_extinct['Clean_Seq']]
tensor_seq_ext = torch.tensor(ext_seq_list, dtype=torch.long)
tensor_target_ext = torch.tensor(df_extinct['TARGET_ID'].values, dtype=torch.long)

# 4. Labels (Log2 AND Z-Score using TRAINING Stats)
y_ext_log2 = np.log2(df_extinct['True_MIC'].values + 1e-6)
y_ext_normalized = (y_ext_log2 - Y_MEAN) / Y_STD
tensor_y_ext = torch.tensor(y_ext_normalized, dtype=torch.float32)

test_dataset = TensorDataset(tensor_seq_ext, tensor_target_ext, tensor_phys_ext, tensor_y_ext)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"✅ Extinct Data: {len(test_dataset)} samples (Z-Scored)")


# --- 2. UPDATED EXPERIMENT FUNCTION ---
class APEXHybrid(nn.Module):
    def __init__(self, vocab_size, num_targets, num_phys_features, embedding_dim=128, hidden_dim=128):
        super(APEXHybrid, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.gru_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.W_att1 = nn.Linear(hidden_dim + embedding_dim, MAX_LEN)
        self.W_att2 = nn.Linear(hidden_dim, 1)
        self.fc_seq = nn.Linear(hidden_dim, hidden_dim)

        self.target_embedding = nn.Embedding(num_targets, 32)
        self.phys_layer = nn.Linear(num_phys_features, 32)

        total_dim = hidden_dim + 32 + 32
        self.fc_1 = nn.Linear(total_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Linear(64, 1)

    def forward(self, seq_input, target_input, phys_input):
        x = self.embedding(seq_input)
        rnn_out, _ = self.gru(x)
        h_rnn = self.layer_norm(self.gru_proj(rnn_out))
        cat_features = torch.cat([h_rnn, x], dim=2)

        a1 = F.softmax(self.W_att1(cat_features), dim=2)
        h_att1 = torch.bmm(a1, h_rnn)
        a2 = F.softmax(self.W_att2(h_att1), dim=1)
        h_att2 = torch.bmm(a2.transpose(1, 2), h_att1).squeeze(1)
        seq_vec = self.fc_seq(h_att2)

        target_vec = self.target_embedding(target_input)
        phys_vec = F.relu(self.phys_layer(phys_input))

        combined = torch.cat([seq_vec, target_vec, phys_vec], dim=1)
        z = self.dropout(F.relu(self.fc_1(combined)))
        return self.fc_out(z)

def run_hybrid_experiment_smart(seed, X_seq, X_targets, X_phys, y, patience=5, max_epochs=100, save_dir="."):
    print(f"\n--- Hybrid Seed {seed} ---")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)

    indices = np.arange(len(y))
    try:
        strat = X_targets.cpu().numpy()
        train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=seed, stratify=strat)
    except:
        train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=seed)

    train_ds = TensorDataset(X_seq[train_idx], X_targets[train_idx], X_phys[train_idx], y[train_idx])
    val_ds = TensorDataset(X_seq[val_idx], X_targets[val_idx], X_phys[val_idx], y[val_idx])
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    num_phys = X_phys.shape[1]

    model = APEXHybrid(
        vocab_size=len(aa_vocab)+1,
        num_targets=len(target_map),
        num_phys_features=num_phys
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float('inf')
    best_weights = None
    counter = 0

    for epoch in range(max_epochs):
        model.train()
        for s, t, p, label in train_loader:
            s, t, p, label = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE), label.to(DEVICE)
            optimizer.zero_grad()
            out = model(s, t, p)
            loss = criterion(out.squeeze(), label)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for s, t, p, label in val_loader:
                s, t, p, label = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE), label.to(DEVICE)
                out = model(s, t, p)
                val_loss += criterion(out.squeeze(), label).item() * s.size(0)

        avg_val = val_loss / len(val_loader.dataset)

        if avg_val < best_val:
            best_val = avg_val
            best_weights = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience: break

    # Save
    save_path = os.path.join(save_dir, f"apex_hybrid_seed_{seed}.pth")
    torch.save(best_weights, save_path)

    model.load_state_dict(best_weights)
    model.eval()
    return model

# --- 3. RUN LOOP & INVERSE TRANSFORM ---
results = {'rho': [], 'preds': [], 'trues': []}
print(f"\n🚀 Starting 3-Seed Evaluation...")

for seed in SEEDS:
    model = run_hybrid_experiment_smart(
        seed, X_seq_padded, X_targets, X_phys, y,
        patience=5, max_epochs=50, save_dir=SAVE_DIR
    )

    # Predict on EXTINCT Data
    seed_preds_z, seed_trues_z = [], []
    with torch.no_grad():
        for s, t, p, label in test_loader:
            s, t, p = s.to(DEVICE), t.to(DEVICE), p.to(DEVICE)
            out = model(s, t, p).squeeze()
            seed_preds_z.extend(out.cpu().numpy())
            seed_trues_z.extend(label.numpy())

    # INVERSE TRANSFORM (Z-Score -> Log2 MIC)
    seed_preds_log2 = np.array(seed_preds_z) * Y_STD + Y_MEAN
    seed_trues_log2 = np.array(seed_trues_z) * Y_STD + Y_MEAN

    rho, _ = spearmanr(seed_trues_log2, seed_preds_log2)
    print(f"   [Seed {seed}] Extinct Spearman: {rho:.4f}")
    results['rho'].append(rho)
    results['preds'].append(seed_preds_log2)
    results['trues'] = seed_trues_log2 # Constant across seeds

# --- 4. GRAPH (Log2 Scale) ---
avg_preds = np.mean(results['preds'], axis=0)
y_true_log2 = results['trues']
final_rho, _ = spearmanr(y_true_log2, avg_preds)

print(f"\nFINAL ENSEMBLE SPEARMAN: {final_rho:.4f}")

plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_true_log2, y=avg_preds, alpha=0.6, s=60, color='darkblue')
min_val, max_val = min(y_true_log2), max(y_true_log2)
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')
plt.title(f"Extinct Data (3-Seed Ensemble)\nSpearman = {final_rho:.3f}")
plt.xlabel("True MIC (Log2)")
plt.ylabel("Predicted MIC (Log2)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Interpretation

| Setting | Spearman rho |
|---|---|
| In-distribution (modern peptides) | **0.7876** |
| Out-of-distribution (extinct peptides) | **0.3564** |

Performance drops by more than half. Per-seed OOD rho ranged from 0.2465 to 0.3866 — high
variance on only 690 points. This is meaningfully better than chance but far from usable, and
the parity plots show heavy regression to the mean.

Tree baselines on the same extinct set: Random Forest rho = 0.3300, XGBoost rho = 0.3305. The
hybrid network is marginally ahead of both.

![Extinct-set tree baselines](../docs/figures/14_extinct_baselines_rf_xgb.png)

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. DATA PREP ---
# We found the variable name in your code: 'df_test_ready'
if 'df_test_ready' not in locals():
    raise NameError("Could not find 'df_test_ready'. Make sure you ran the cell that loads the data!")

df_test_raw = df_test_ready.copy()

# Use baseline_df for training
df_train_raw = baseline_df.copy()

ignore_cols = ['ID', 'SEQUENCE', 'MIC', 'is_AMP', 'log_MIC', 'TARGET ID']

# Prep Train
X_train = df_train_raw.drop(columns=[c for c in ignore_cols if c in df_train_raw.columns])
y_train = df_train_raw['log_MIC']

# Prep Test
X_test = df_test_raw.drop(columns=[c for c in ignore_cols if c in df_test_raw.columns])
# Note: Your snippet implied the column in test might be 'True_MIC', not 'log_MIC'
# We will check for both.
if 'log_MIC' in df_test_raw.columns:
    y_test = df_test_raw['log_MIC']
elif 'True_MIC' in df_test_raw.columns:
    # Convert True_MIC to log2 if needed, matching your snippet's logic
    y_test = np.log2(df_test_raw['True_MIC'] + 1e-6)
else:
    raise KeyError("Could not find 'log_MIC' or 'True_MIC' in df_test_ready")

# One-Hot Encode Targets (if they exist)
if 'TARGET ID' in df_train_raw.columns:
    dummies_train = pd.get_dummies(df_train_raw['TARGET ID'], prefix='Target', dtype=float)
    X_train = pd.concat([X_train, dummies_train], axis=1)

    # Check if Test has TARGET_ID or TARGET ID
    tgt_col = 'TARGET ID' if 'TARGET ID' in df_test_raw.columns else 'TARGET_ID'
    if tgt_col in df_test_raw.columns:
        dummies_test = pd.get_dummies(df_test_raw[tgt_col], prefix='Target', dtype=float)
        X_test = pd.concat([X_test, dummies_test], axis=1)

# FORCE ALIGNMENT: Ensure Test has exactly the same columns as Train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# --- 2. EVALUATION FUNCTION ---
def evaluate_on_extinct(model_name, model_class, params):
    seeds = [42, 123, 2024]
    ensemble_preds = np.zeros(len(y_test))

    print(f"\n--- Testing {model_name} ---")

    for seed in seeds:
        model = model_class(**params, random_state=seed, n_jobs=-1)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        ensemble_preds += preds

    ensemble_preds /= len(seeds)
    final_rho, _ = spearmanr(y_test, ensemble_preds)

    # Graphing
    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, ensemble_preds, alpha=0.5, color='navy', s=30)
    plt.plot([0, 8], [0, 8], 'r--', label='Perfect')
    plt.title(f"{model_name}\nSpearman = {final_rho:.4f}")
    plt.xlabel("True MIC (Log2)"); plt.ylabel("Predicted MIC (Log2)")
    plt.grid(True, alpha=0.3)
    plt.show()

    return final_rho

# --- 3. RUN SHOWDOWN ---
rf_score = evaluate_on_extinct("Random Forest", RandomForestRegressor, {'n_estimators': 100, 'max_depth': 20})
xgb_score = evaluate_on_extinct("XGBoost", XGBRegressor, {'n_estimators': 200, 'learning_rate': 0.05})

print("\nFINAL SPEARMAN (Extinct Data)")
print(f"Random Forest: {rf_score:.4f}")
print(f"XGBoost:       {xgb_score:.4f}")

## 7. Does AlphaFold structure help?

The central question of the project. Five per-peptide structural summaries are computed from
AlphaFold2 predictions (see notebooks 02-05):

| Feature | Meaning |
|---|---|
| `pLDDT_mean` | AlphaFold's own per-residue confidence, averaged |
| `Frac_Helix` | Fraction of residues in the alpha-helical Ramachandran region (phi/psi windows) |
| `Frac_Sheet` | Fraction of residues in the beta-sheet Ramachandran region (phi/psi windows) |
| `Backbone_Rigidity` | Mean circular variance of the phi and psi angle distributions |
| `Avg_Degree` | Mean residue contact-map degree |

**Method note:** this comparison uses a Random Forest trained on *frozen embeddings* extracted
from the already-trained hybrid network, concatenated with the structural features. It is a
different model family from the end-to-end network used in sections 2-6, so its absolute
numbers are not comparable to those above — only the three variants within this section are
comparable to each other.

In [ ]:
# --- CHUNK 1: SETUP HELPER FUNCTION (FIXED FOR APEXHYBRID) ---
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.nn.utils.rnn import pad_sequence

print("⚙️ CHUNK 1: Setting up Fixed Feature Extractor...")

def get_embeddings_from_list(model, sequences, vocab, max_len, device):
    """
    Manually runs sequences through the APEXHybrid sequence branch.
    """
    model.eval()
    tokenized = []

    # 1. Tokenize
    for seq in sequences:
        s = str(seq) if pd.notnull(seq) else ""
        tokens = [vocab.get(c, 0) for c in s]
        tokenized.append(torch.tensor(tokens, dtype=torch.long))

    if len(tokenized) == 0:
        return np.array([])

    # 2. Pad
    padded_seqs = pad_sequence(tokenized, batch_first=True, padding_value=0)

    # Fix length
    if padded_seqs.shape[1] > max_len:
        padded_seqs = padded_seqs[:, :max_len]
    elif padded_seqs.shape[1] < max_len:
        padding = torch.zeros((padded_seqs.shape[0], max_len - padded_seqs.shape[1]), dtype=torch.long)
        padded_seqs = torch.cat([padded_seqs, padding], dim=1)

    padded_seqs = padded_seqs.to(device)

    # 3. EXTRACT FEATURES (Manual Forward Pass)
    with torch.no_grad():
        # A. Embed
        x = model.embedding(padded_seqs)

        # B. GRU
        rnn_out, _ = model.gru(x)
        h_rnn = model.layer_norm(model.gru_proj(rnn_out))

        # C. Attention 1
        cat_features = torch.cat([h_rnn, x], dim=2)
        # Note: W_att1 is a layer in your APEXHybrid class
        a1 = F.softmax(model.W_att1(cat_features), dim=2)
        h_att1 = torch.bmm(a1, h_rnn)

        # D. Attention 2
        a2 = F.softmax(model.W_att2(h_att1), dim=1)
        h_att2 = torch.bmm(a2.transpose(1, 2), h_att1).squeeze(1)

        # E. Final Projection (The 128D Vector)
        embeddings = model.fc_seq(h_att2)

    return embeddings.cpu().numpy()

print("✅ Chunk 1 Fixed. Now run Chunk 5 again!")

In [ ]:
# --- CHUNK 2: LOAD ALPHAFOLD DATA (DEBUG MODE) ---
print("\n📂 CHUNK 2: Loading AlphaFold CSVs...")

try:
    # 1. Load Files
    df_af_train = pd.read_csv(STRUCT_TRAIN_CSV)
    df_af_test = pd.read_csv(STRUCT_TEST_CSV)

    print(f"   - Raw Train Rows: {len(df_af_train)}")
    print(f"   - Raw Test Rows:  {len(df_af_test)}")

    # 2. Clean IDs & Debug
    # Train
    df_af_train['ID'] = df_af_train['protein_name'].str.replace('.result', '', regex=False).astype(int)
    print(f"   - Train IDs example: {df_af_train['ID'].head(3).tolist()}")

    # Test
    df_af_test['ID'] = df_af_test['protein_name'].str.replace('.result', '', regex=False)
    print(f"   - Test IDs example:  {df_af_test['ID'].head(3).tolist()}")

    # 3. Select Columns
    af_cols = ['pLDDT_mean', 'Frac_Helix', 'Frac_Sheet', 'Backbone_Rigidity', 'Avg_Degree']

    # Check for NaNs
    if df_af_train[af_cols].isnull().any().any():
        print("   ⚠️ Warning: NaNs found in AlphaFold Train Data. Filling with 0.")
        df_af_train = df_af_train.fillna(0)

    df_af_train = df_af_train[['ID'] + af_cols]
    df_af_test = df_af_test[['ID'] + af_cols]

    print(f"✅ Chunk 2 Complete.")
    print(f"   - Ready for Merge: {len(df_af_train)} Train, {len(df_af_test)} Test")

except FileNotFoundError as e:
    print(f"❌ ERROR: Could not find file. {e}")
except Exception as e:
    print(f"❌ ERROR in Chunk 2: {e}")

In [ ]:
# --- CHUNK 3: MERGE TRAINING DATA (SAFE MODE) ---
print("\n🔗 CHUNK 3: Merging Training Data...")

if 'baseline_df' not in locals():
    print("❌ ERROR: 'baseline_df' is missing. Run the top of your notebook.")
else:
    # 1. SEARCH AND RESCUE for the 'ID'
    # Check if ID is a column
    if 'ID' in baseline_df.columns:
        print("   - ✅ Found 'ID' in columns.")
    # Check if ID is the Index
    elif baseline_df.index.name == 'ID':
        print("   - ⚠️ Found 'ID' in Index. Resetting index to make it a column...")
        baseline_df = baseline_df.reset_index()
    # Check if Index has no name but looks like IDs
    else:
        print("   - ⚠️ 'ID' column missing. Assuming the Index contains the IDs...")
        baseline_df = baseline_df.reset_index()
        # Rename the new 'index' column to 'ID' if needed
        if 'index' in baseline_df.columns:
            baseline_df = baseline_df.rename(columns={'index': 'ID'})
            print("   - Renamed 'index' column to 'ID'.")

    # 2. VERIFY IDs ARE COMPATIBLE
    # Ensure IDs are integers (since AlphaFold IDs are integers like 16088)
    try:
        baseline_df['ID'] = baseline_df['ID'].astype(int)
        print("   - IDs converted to Integer format.")
    except:
        print("   - ⚠️ Warning: IDs could not be converted to int. Merge might fail if formats differ.")

    # 3. PERFORM MERGE
    # This will broadcast the 1 AlphaFold structure to ALL rows for that peptide
    train_hybrid = pd.merge(baseline_df, df_af_train, on='ID', how='inner')

    # 4. REPORT
    n_unique_ids = train_hybrid['ID'].nunique()
    n_total_rows = len(train_hybrid)

    if n_total_rows == 0:
        print("❌ CRITICAL FAILURE: 0 matches found.")
        print(f"   - Baseline Sample IDs: {baseline_df['ID'].head(3).tolist()}")
        print(f"   - AlphaFold Sample IDs: {df_af_train['ID'].head(3).tolist()}")
    else:
        print(f"✅ Chunk 3 Complete.")
        print(f"   - Unique Peptides with Structure: {n_unique_ids}")
        print(f"   - Total Training Rows (Melted):   {n_total_rows}")
        print(f"   - Multiplier Effect: ~{n_total_rows/n_unique_ids if n_unique_ids else 0:.1f} rows per peptide")

In [ ]:
# --- CHUNK 4: MERGE EXTINCT DATA (CORRECTED) ---
print("\n🧬 CHUNK 4: Reloading & Merging Extinct Data...")

try:
    # 1. LOAD THE EXTINCT DATA FILE
    # We use the path from your notebook variables
    if 'TEST_FILE' not in locals():
        TEST_FILE = EXTINCT_XLSX

    print(f"   - Reading file: {TEST_FILE}")
    # Using header=1 because your CSV shows the header is on the second line
    try:
        raw_extinct = pd.read_excel(TEST_FILE, header=1)
    except:
        # Fallback if it's actually a CSV in your drive
        raw_extinct = pd.read_csv(TEST_FILE, header=1)

    print(f"   - Loaded {len(raw_extinct)} rows.")

    # 2. EXTRACT IDs FROM THE 'Peptide' COLUMN
    # We look for the column named 'Peptide' (as seen in your CSV)
    id_col = 'Peptide'
    if id_col not in raw_extinct.columns:
        print(f"   ⚠️ 'Peptide' column not found. Columns are: {raw_extinct.columns.tolist()}")
        # Fallback search
        for col in raw_extinct.columns:
            if 'ANT45524' in raw_extinct[col].astype(str).values:
                id_col = col
                print(f"   - Found target ID in column: '{col}'")
                break

    # Create a clean 'ID' column for merging
    # Logic: "ANT45524.1-VFL13" -> "ANT45524"
    print(f"   - Extracting IDs from column '{id_col}'...")
    raw_extinct['ID'] = raw_extinct[id_col].astype(str).str.split('.').str[0]

    # 3. MERGE WITH ALPHAFOLD
    # Ensure df_af_test is loaded (from Chunk 2)
    if 'df_af_test' not in locals():
         raise ValueError("df_af_test is missing! Run Chunk 2 first.")

    print(f"   - Merging with AlphaFold data...")
    test_hybrid = pd.merge(raw_extinct, df_af_test, on='ID', how='inner')

    if len(test_hybrid) == 0:
        print("❌ STOPPING: 0 Matches. Check your ID extraction logic.")
        print(f"   - Excel Sample IDs: {raw_extinct['ID'].head().tolist()}")
        print(f"   - AF Sample IDs:    {df_af_test['ID'].head().tolist()}")
    else:
        # 4. PREPARE FINAL TEST SET
        # Clean Sequence (Column 'Seq' in your CSV)
        test_hybrid['Clean_Seq'] = test_hybrid['Seq'].str.upper().str.replace('[^A-Z]', '', regex=True)

        # Identify bacteria columns for MIC targets (Columns with 'ATCC', 'PAO1', etc.)
        bacteria_cols = [c for c in raw_extinct.columns if 'ATCC' in c or 'PAO1' in c or 'MRSA' in c]
        print(f"   - Found {len(bacteria_cols)} bacterial targets.")

        # Melt to get MICs
        test_hybrid_long = test_hybrid.melt(
            id_vars=['ID', 'Clean_Seq'] + af_cols,
            value_vars=bacteria_cols,
            value_name='True_MIC'
        )

        # Clean MIC values (handle '>', '<', 'N.A.')
        test_hybrid_long['True_MIC'] = pd.to_numeric(
            test_hybrid_long['True_MIC'].astype(str).replace(['N.A.', 'nan', '>', '<'], ['128', '128', '', ''], regex=True),
            errors='coerce'
        ).fillna(128)

        # Group by ID to get Unique Peptides (One prediction per peptide)
        # We take the FIRST valid MIC we find, or you could average them.
        test_hybrid_unique = test_hybrid_long.groupby('ID').first().reset_index()

        print(f"✅ Chunk 4 Complete.")
        print(f"   - Successfully matched {len(test_hybrid_unique)} extinct peptides with 3D structure.")

except Exception as e:
    print(f"❌ Chunk 4 FAILED: {e}")

In [ ]:
# --- CHUNK 5: FINAL TRAINING (WITH SPEARMAN) ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import numpy as np

print("\n🌲 CHUNK 5: Training Final Hybrid Model...")

if 'train_hybrid' in locals() and 'test_hybrid_unique' in locals() and len(train_hybrid) > 0:

    # 1. PREPARE DATA
    # Train Data (Mined Peptides)
    X_train_seq = get_embeddings_from_list(model, train_hybrid['SEQUENCE'].values, vocab, MAX_SEQ_LENGTH, DEVICE)
    X_train_str = train_hybrid[af_cols].fillna(0).values
    X_train_full = np.hstack([X_train_seq, X_train_str])
    y_train = np.log2(train_hybrid['MIC'].values + 1e-6)

    # Test Data (Extinct Peptides)
    X_test_seq = get_embeddings_from_list(model, test_hybrid_unique['Clean_Seq'].values, vocab, MAX_SEQ_LENGTH, DEVICE)
    X_test_str = test_hybrid_unique[af_cols].fillna(0).values
    X_test_full = np.hstack([X_test_seq, X_test_str])
    y_test = np.log2(test_hybrid_unique['True_MIC'].values + 1e-6)

    print(f"   - Train Shape: {X_train_full.shape}")
    print(f"   - Test Shape:  {X_test_full.shape}")

    # 2. TRAIN MODELS (COMPARISON)
    # A. Sequence Only (Baseline)
    rf_seq = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_seq.fit(X_train_seq, y_train)
    preds_seq = rf_seq.predict(X_test_seq)
    r2_seq = r2_score(y_test, preds_seq)
    rho_seq, _ = spearmanr(y_test, preds_seq)

    # B. Structure Only
    rf_str = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_str.fit(X_train_str, y_train)
    preds_str = rf_str.predict(X_test_str)
    r2_str = r2_score(y_test, preds_str)
    rho_str, _ = spearmanr(y_test, preds_str)

    # C. Hybrid (The Goal)
    rf_hy = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_hy.fit(X_train_full, y_train)
    preds_hy = rf_hy.predict(X_test_full)
    r2_hy = r2_score(y_test, preds_hy)
    rho_hy, _ = spearmanr(y_test, preds_hy)

    # 3. REPORT
    print("\n" + "="*50)
    print(f"🏆 FINAL RESULTS (Extinct Peptides)")
    print(f"{'Model':<20} | {'R2 Score':<10} | {'Spearman (Rho)':<10}")
    print("-" * 50)
    print(f"{'1. Sequence Only':<20} | {r2_seq:.4f}     | {rho_seq:.4f}")
    print(f"{'2. Structure Only':<20} | {r2_str:.4f}     | {rho_str:.4f}")
    print(f"{'3. HYBRID MODEL':<20} | {r2_hy:.4f}     | {rho_hy:.4f}")
    print("="*50)

    if r2_hy > r2_seq or rho_hy > rho_seq:
        print("✅ SUCCESS: Hybrid model shows improvement!")
    else:
        print("⚠️ NOTE: Hybrid model performed similarly to baseline.")

else:
    print("❌ Cannot train. Check Chunks 3 and 4.")

### Result: it does not help

| Model | R2 | Spearman rho |
|---|---|---|
| Sequence only | -0.6985 | **0.2010** |
| Structure only | -1.0162 | 0.1466 |
| Sequence + structure (hybrid) | -0.6719 | 0.1824 |

![AlphaFold hybrid results and feature importance](../docs/figures/15_alphafold_hybrid_results_and_importance.png)

Adding structure improves R2 slightly (-0.699 -> -0.672) but makes **rank correlation worse**
(0.201 -> 0.182). Since rank order is what matters for screening, this is not an improvement.

> **Note on the original output.** The cell above prints `SUCCESS: Hybrid model shows
> improvement!` — its condition is `r2_hy > r2_seq or rho_hy > rho_seq`, an `or`, so the R2
> gain alone triggers the message even though rho regressed. The printed banner should be
> disregarded; the table is the honest read. Negative R2 across all three variants means every
> variant is worse than predicting the training mean.

Feature-importance attribution makes the point sharply:

**Sequence information: 99.1% | Structural information: 0.9%**

In [ ]:
# --- CHUNK 6: VISUALIZATION (WITH SPEARMAN) ---
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("📊 CHUNK 6: Generating Graphs...")

# Check if Chunk 5 ran
if 'r2_seq' in locals() and 'rho_seq' in locals():

    # Set up the figure
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # ---------------------------
    # GRAPH 1: MODEL COMPARISON (Grouped Bar)
    # ---------------------------
    models = ['Sequence\nOnly', 'Structure\nOnly', 'Hybrid\n(Seq+3D)']
    r2_scores = [r2_seq, r2_str, r2_hy]
    rho_scores = [rho_seq, rho_str, rho_hy]

    x = np.arange(len(models))
    width = 0.35

    # Create grouped bars
    bars1 = axes[0].bar(x - width/2, r2_scores, width, label='R² (Accuracy)', color='#bdc3c7')
    bars2 = axes[0].bar(x + width/2, rho_scores, width, label='Spearman (Ranking)', color='#2ecc71')

    axes[0].set_title('Performance Metrics', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Score')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(models)
    axes[0].legend()
    axes[0].set_ylim(min(min(r2_scores), min(rho_scores)) - 0.1, max(max(r2_scores), max(rho_scores)) + 0.1)

    # Add text labels on bars
    for b in bars1 + bars2:
        h = b.get_height()
        axes[0].text(b.get_x() + b.get_width()/2., h, f'{h:.2f}',
                     ha='center', va='bottom', fontsize=10, fontweight='bold')

    # ---------------------------
    # GRAPH 2: PREDICTED vs ACTUAL (Hybrid)
    # ---------------------------
    # Using existing predictions
    preds_hy = rf_hy.predict(X_test_full)

    axes[1].scatter(y_test, preds_hy, alpha=0.6, color='#2ecc71', edgecolor='k')

    # Draw perfect fit line
    min_val = min(y_test.min(), preds_hy.min())
    max_val = max(y_test.max(), preds_hy.max())
    axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

    axes[1].set_title(f'Hybrid Model Accuracy\nSpearman Rho: {rho_hy:.3f}', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('True Log(MIC)')
    axes[1].set_ylabel('Predicted Log(MIC)')
    axes[1].legend()

    # ---------------------------
    # GRAPH 3: FEATURE IMPORTANCE
    # ---------------------------
    importances = rf_hy.feature_importances_
    feat_names = [f"Seq_Emb_{i}" for i in range(128)] + af_cols

    feat_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
    feat_df = feat_df.sort_values('Importance', ascending=False).head(10)

    bar_colors = ['#3498db' if 'Seq' in x else '#e67e22' for x in feat_df['Feature']]

    sns.barplot(x='Importance', y='Feature', data=feat_df, ax=axes[2], palette=bar_colors)
    axes[2].set_title('Top 10 Features', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Importance')

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#3498db', label='Sequence'),
                       Patch(facecolor='#e67e22', label='3D Structure')]
    axes[2].legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.show()

    print("✅ Graphs Generated!")

else:
    print("❌ Error: Variables from Chunk 5 not found.")

In [ ]:
# --- CHUNK 7: SANITY CHECK ---
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 INSPECTING 3D DATA...")

# 1. Check if the features actually vary
# We look at the Training Set (Mined)
print(train_hybrid[af_cols].describe())

# 2. Visual Check
# Plot the distribution of Helix Fraction
plt.figure(figsize=(10, 4))
sns.histplot(train_hybrid['Frac_Helix'], bins=20, kde=True, color='orange')
plt.title('Distribution of Helix Fraction in Training Data')
plt.xlabel('Helix Fraction (0.0 - 1.0)')
plt.show()

# 3. Correlation Check
# Does Helix Fraction correlate with MIC?
# If correlation is near 0, the feature is useless for this specific task.
corr = train_hybrid['Frac_Helix'].corr(train_hybrid['MIC'])
print(f"\nCorrelation between Helix Fraction and MIC: {corr:.4f}")

### Why structure contributed so little

![Helix fraction vs MIC](../docs/figures/16_helix_fraction_vs_mic.png)

Correlation between helix fraction and MIC: **0.0152** — essentially zero.

Inspecting the structural features across 2,828 peptides:

```
        pLDDT_mean   Frac_Helix   Frac_Sheet  Backbone_Rigidity   Avg_Degree
mean     74.776709     0.116433     0.033590           0.133545     4.750076
std       7.203278     0.277758     0.103917           0.125791     1.304558
50%      74.512000     0.000000     0.000000           0.096835     4.400000
```

Two things stand out. `Frac_Helix` and `Frac_Sheet` have a median of exactly 0 — most predicted
structures have no assigned secondary structure at all. And mean pLDDT is ~75 with a standard
deviation of only 7, so the features barely vary across peptides.

This is mechanistically unsurprising. Antimicrobial peptides are typically disordered in
solution and only fold into amphipathic helices *on contact with a membrane*. AlphaFold2
predicts a single static structure in isolation, which is close to the wrong conformational
state for this problem, and it predicts it with low confidence. The information that matters —
membrane-bound conformation, amphipathic moment, insertion depth — is not in these features.

## Summary

- A three-arm sequence + physicochemical + target encoder reaches **R2 = 0.63, Spearman rho = 0.79**
  in-distribution, modestly ahead of Random Forest (R2 ~ 0.61) and clearly ahead of XGBoost (R2 ~ 0.43).
- On 69 extinct peptides the same model reaches **Spearman rho = 0.36** — a drop of more than half.
- **AlphaFold2 structural features did not help**, receiving 0.9% of total feature attribution and
  slightly degrading rank correlation.

See [`../docs/RESULTS.md`](../docs/RESULTS.md) for the full results table and the known
limitations, including a peptide-level leakage caveat that affects how the in-distribution
numbers should be read.